# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR^2 dataset Croissant package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Set Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review all available record sets and their associated fields (columns). All entities are referenced by their `@id`.

In [ ]:
# List all record sets' @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset's main 'recordSet' property. Searching by iterating dataset.record_sets...")

record_set_ids = []
for rs in dataset.record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")
    print()

if not record_set_ids:
    print("No record sets found.")

## 3. Data Extraction
For demonstration, we will extract all available record sets into `pandas` DataFrames, indexed by their `@id`.

Replace `record_set_id_to_demo` with the `@id` of a record set you wish to inspect in detail if multiple are present.

In [ ]:
# Extract all record sets as DataFrames keyed by @id
dataframes = {}
for rid in record_set_ids:
    # Records come as generators — convert to a list of dict for DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=rid)))
    dataframes[rid] = df
    print(f"Loaded DataFrame for record set {rid} (shape: {df.shape})")

# For demonstration, select the first record set
if len(record_set_ids) > 0:
    record_set_id_to_demo = record_set_ids[0]
    print(f"Example record set: {record_set_id_to_demo}")
    print("\nFields (@id):", list(dataframes[record_set_id_to_demo].columns))
    display(dataframes[record_set_id_to_demo].head())
else:
    print("No record sets available to extract.")

## 4. Exploratory Data Analysis (EDA)
In this section, we'll select a numeric field by its `@id`, filter for records greater than a threshold, normalize the numeric values, and (where available) group by another field using its `@id`. All operations reference fields by their unique `@id`.

In [ ]:
# Identify numeric fields (@id) in the chosen record set
demo_df = dataframes[record_set_id_to_demo]
numeric_fields = demo_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields in '{record_set_id_to_demo}': {numeric_fields}")

if not numeric_fields:
    print("No numeric fields available for analysis in the selected record set.")
else:
    # Choose the first numeric field for demo
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field '@id': {numeric_field_id}")

    # Set a threshold for EDA filtering.
    threshold = demo_df[numeric_field_id].quantile(0.75)  # Top quartile as example
    filtered_df = demo_df[demo_df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered {len(filtered_df)} records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized column '{norm_col}' (z-score):")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical column
    cat_fields = demo_df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for col in cat_fields:
        if demo_df[col].nunique() > 1 and demo_df[col].nunique() < demo_df.shape[0]:
            group_field = col
            break

    if group_field:
        print(f"\nGrouping by field '@id': {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print("Average of numeric field by group:")
        display(grouped_df)
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Show a histogram of the selected numeric field and a boxplot grouped by the chosen categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric fields available to plot.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(demo_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=demo_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated programmatic, schema-driven access to a clinical oncology dataset using `mlcroissant`.
- We showed how to discover record set and field `@id`s, load data into DataFrames, filter, normalize, and analyze by those keys.
- All operations were referenced strictly by `@id` to ensure reproducibility and stability against schema changes.
- This workflow can be adapted to any other dataset described by a Croissant schema.
